# SaaS Raw Inventory And Column Selection

이 notebook은 `saas_subscription_churn` 원본에서
**현재 고객 상태를 보고 앞으로 30일 안에 떠날지**를 보기 위해 어떤 raw 컬럼이 필요한지 정하는 시작 notebook이다.

즉 여기서는 모델을 만들지 않고,
아래 질문에 답하기 위해 원본에서 무엇을 가져와야 하는지 정리한다.

- 현재 고객 상태 중 subscription / usage 관련 어떤 신호를 볼 것인가
- 원본에서 renewal 관련 신호를 직접 보거나 추론할 수 있는가
- 원본에서 최종적으로 고객이 떠났다는 라벨을 만들 수 있는가


## Part Guide

- Part 1. Environment and raw loading
  - dataset 위치를 확인하고 실제 csv를 로딩한다.
- Part 2. Table-level inventory
  - table별 row 수 / column 수 / date 범위를 확인한다.
- Part 3. Column-level inventory
  - column별 dtype / non-null ratio / sample value를 본다.
- Part 4. 우리가 보고 싶은 질문 정리
  - 현재 상태 -> 30일 내 이탈 여부라는 질문을 고정한다.
- Part 5. 원본 컬럼 -> 최종 컬럼 매핑 정리
  - 원본에 뭐가 있고, 그래서 뭘 남길지 적는다.
- Part 6. Handoff to panel builder
  - 다음 notebook에서 어떤 row와 target을 만들지 연결한다.

In [ ]:
from pathlib import Path

import pandas as pd

CWD = Path.cwd().resolve()
WORKSPACE_ROOT = next(
    (path for path in [CWD, *CWD.parents] if (path / 'back_research').exists() and (path / 'docs').exists()),
    CWD,
)
DATASET_DIR = WORKSPACE_ROOT / 'back_research' / 'wonbeenlee' / 'datasets' / 'saas_subscription_churn'

print(DATASET_DIR)
print(DATASET_DIR.exists())

## Part 1. Environment and Raw Loading

이 part에서 하는 일:

- 실제 csv를 읽는다.
- date/datetime 컬럼을 parse한다.
- 이후 inventory와 column selection의 입력을 만든다.

In [ ]:
def load_saas_tables(dataset_dir: Path) -> dict[str, pd.DataFrame]:
    tables = {
        'accounts': ('ravenstack_accounts.csv', ['signup_date']),
        'subscriptions': ('ravenstack_subscriptions.csv', ['start_date', 'end_date']),
        'feature_usage': ('ravenstack_feature_usage.csv', ['usage_date']),
        'support_tickets': ('ravenstack_support_tickets.csv', ['submitted_at', 'closed_at']),
        'churn_events': ('ravenstack_churn_events.csv', ['churn_date']),
    }
    loaded = {}
    for name, (filename, parse_dates) in tables.items():
        loaded[name] = pd.read_csv(dataset_dir / filename, parse_dates=parse_dates)
    return loaded

saas_tables = load_saas_tables(DATASET_DIR)
list(saas_tables.keys())

## Part 2. Table-Level Inventory

이 part에서 하는 일:

- table별 실제 row 수, column 수를 확인한다.
- time-aware table의 min/max date도 같이 본다.

왜 필요한가:

- 어떤 table이 panel row의 backbone이 될지, 어떤 table이 보조 table인지 먼저 감을 잡아야 한다.
- row volume과 날짜 범위를 보면 어떤 aggregation이 가능한지 바로 드러난다.

In [ ]:
def build_table_inventory(tables: dict[str, pd.DataFrame]) -> pd.DataFrame:
    rows = []
    for table_name, frame in tables.items():
        date_cols = [col for col in frame.columns if 'date' in col or col.endswith('_at')]
        min_time = None
        max_time = None
        for col in date_cols:
            if pd.api.types.is_datetime64_any_dtype(frame[col]):
                series = frame[col].dropna()
                if not series.empty:
                    col_min = series.min()
                    col_max = series.max()
                    min_time = col_min if min_time is None else min(min_time, col_min)
                    max_time = col_max if max_time is None else max(max_time, col_max)
        rows.append({
            'table_name': table_name,
            'row_count': len(frame),
            'column_count': len(frame.columns),
            'date_columns': ', '.join(date_cols),
            'min_time': min_time,
            'max_time': max_time,
        })
    return pd.DataFrame(rows)

saas_table_inventory = build_table_inventory(saas_tables)
saas_table_inventory

## Part 3. Column-Level Inventory

이 part에서 하는 일:

- column별 dtype, non-null ratio, sample value를 실제 raw 기준으로 본다.
- 문서 기억이 아니라 현재 schema 그대로 판단한다.

In [ ]:
def build_column_inventory(tables: dict[str, pd.DataFrame]) -> pd.DataFrame:
    rows = []
    for table_name, frame in tables.items():
        for column in frame.columns:
            non_null_ratio = float(frame[column].notna().mean())
            sample_series = frame[column].dropna()
            sample_value = sample_series.iloc[0] if not sample_series.empty else None
            rows.append({
                'table_name': table_name,
                'column_name': column,
                'dtype': str(frame[column].dtype),
                'non_null_ratio': round(non_null_ratio, 4),
                'sample_value': sample_value,
            })
    return pd.DataFrame(rows)

saas_column_inventory = build_column_inventory(saas_tables)
saas_column_inventory

## Part 4. 우리가 보고 싶은 질문 정리

raw schema를 본 다음, 이제 우리가 보고 싶은 질문을 먼저 고정한다.

지금 우리가 보고 싶은 건 복잡한 score가 아니라 아주 단순한 질문이다.

- 현재 고객 상태를 보면 앞으로 30일 안에 떠날지

이 질문에 답하려면 아래 현재 상태 신호가 필요하다.

- current plan / billing cadence / current value
- renewal / expiry / cancel pressure
- recent usage intensity와 usage drop

반대로 지금 직접 안 볼 것은 아래다.

- free text feedback
- support quality 중심 신호
- 이미 event 이후를 설명하는 post-hoc 변수
- 세부 feature mix 수준의 미세 분해


## Part 5. 원본 컬럼 -> 최종 컬럼 매핑 정리

이 part에서 하는 일:

- 원본 데이터에 실제로 있는 핵심 raw column을 먼저 적는다.
- 그 raw column들로부터 우리가 1차에서 어떤 신호를 뽑아낼지 적는다.
- 마지막에 이번 1차에서 직접 안 쓸 것들을 정리한다.

이 part는 계산 결과가 아니라 해석 결과이므로 markdown으로 남긴다.

### 1. 원본 데이터에 실제로 있는 핵심 raw column

`accounts`
- `account_id`: account 기준 join key
- `signup_date`: 관계 시작 시점
- `plan_tier`: 가입 시점 초기 plan
- `seats`: 가입 시점 초기 seat 규모
- `is_trial`: trial 여부

`subscriptions`
- `start_date`: subscription 시작
- `end_date`: subscription 종료 또는 만료 기준
- `plan_tier`: billing 시점 plan
- `seats`: subscription 단위 seat 규모
- `mrr_amount`: 월 매출 값
- `arr_amount`: 연환산 매출 값
- `billing_frequency`: monthly / annual cadence
- `auto_renew_flag`: auto-renew 여부
- `upgrade_flag`: 상향 변경 여부
- `downgrade_flag`: 하향 변경 여부
- `churn_flag`: subscription ended 여부

`feature_usage`
- `usage_date`: usage 시점
- `usage_count`: 활동 빈도
- `usage_duration_secs`: 활동 시간
- `error_count`: 사용 마찰 signal
- `feature_name`: 사용 feature 종류

`churn_events`
- `churn_date`: 실제 churn event 시점
- `reason_code`: churn reason code
- `refund_amount_usd`: refund 규모
- `feedback_text`: free-text 피드백

`support_tickets`
- `submitted_at`, `resolution_time_hours`, `satisfaction_score`: support/trust 계열 신호

### 2. 이 raw column들로부터 우리가 최종적으로 만들 것

현재 상태 컬럼
- `days_since_join`
  - `signup_date` 로부터 계산
- `plan_tier`
  - active subscription의 `plan_tier` 에서 읽음
- `billing_frequency`
  - active subscription의 `billing_frequency` 에서 읽음
- `auto_renew_flag`
  - active subscription의 `auto_renew_flag` 에서 읽음
- `mrr_amount`
  - active subscription의 `mrr_amount` 에서 읽음
- `seats`
  - active subscription의 `seats` 에서 읽음
- `days_to_expiry`
  - active subscription의 `end_date` 로부터 계산
- `plan_change_up_flag_90d`, `plan_change_down_flag_90d`
  - 최근 90일 `upgrade_flag`, `downgrade_flag` 집계
- `cancel_flag_30d`
  - 최근 30일 `churn_date` 존재 여부
- `renewal_failure_count_90d`
  - 최근 90일 `end_date` 이후 follow-on subscription 부재 횟수
- `usage_intensity_30d`
  - 최근 30일 `usage_count`, `usage_duration_secs` 집계
- `usage_drop_ratio_30d_vs_prev_30d`
  - 최근 30일과 이전 30일 usage 비교

미래 결과 컬럼
- `leave_next_30d`
  - 다음 30일 안에 `churn_event` 가 있거나 renewal이 안 되면 1
- `leave_reason`
  - `churn_event`, `non_renewal`, `stay` 중 하나

### 3. 이번 1차에서 직접 안 쓸 것

Branch C로 분리할 것
- `support_tickets` 의 support / trust 계열 컬럼

첫 패스에서 제외할 것
- `account_name`: identifier
- `feedback_text`: free text
- `feature_name`: 세부 feature mix는 나중에 검토
- `accounts.churn_flag`: current feature로 쓰면 leakage 위험이 큼

요약하면, 먼저 raw에 실제로 있는 컬럼을 확인하고, 그 다음 그 raw에서 어떤 current-state feature와 future target을 뽑아낼지를 적는 순서로 본다.

## Part 6. Handoff To Panel Builder

이 notebook 다음에 바로 할 일:

1. 위 raw column 중 1차에서 직접 쓰기로 한 것만 가지고 panel builder를 만든다.
2. support / trust 계열은 여기서 직접 feature로 넣지 않고 Branch C 후보로 분리해 둔다.
3. identifier / free text / leakage 위험 컬럼은 first-pass builder에서 제외한다.

즉 다음 notebook은 이 raw-to-derived 정리를 전제로 움직인다.